In [1]:
import os
import numpy as np
import pandas as pd
import os
import json
import scipy.stats as stats
from pathlib import Path
import re

os.chdir(os.getcwd())

## Walk Over Results


In [2]:
dir_path = Path('./results/')

In [3]:
def process_metrics_jsons(directory_path):
    """
    Process JSON files containing training metrics and combine them into a DataFrame.
    Each JSON file contains lists of metrics across epochs.

    Parameters:
    directory_path (str): Path to the directory containing JSON files

    Returns:
    pandas.DataFrame: Combined DataFrame with metrics and their sources
    """
    # Convert directory path to Path object
    dir_path = Path(directory_path)

    # List to store all data
    all_data = []

    # Walk through directory
    for file_path in dir_path.rglob('*.json'):
        try:
            # Read JSON file
            with open(file_path, 'r') as file:
                data = json.load(file)

            # Get the number of epochs (length of any metric list)
            n_epochs = len(data['balanced_accuracy'])

            # Create records for each epoch
            for epoch in range(n_epochs):
                record = {
                    'source_file': file_path.name,
                    'epoch': epoch,
                    'balanced_accuracy': data['balanced_accuracy'][epoch],
                    'val_loss': data['val_loss'][epoch],
                    'val_f1': data['val_f1'][epoch],
                    'val_AUPRC': data['val_AUPRC'][epoch],
                    'val_MCC': data['val_MCC'][epoch]
                }
                all_data.append(record)

        except Exception as e:
            print(f"Error processing {file_path.name}: {str(e)}")
            continue

    if not all_data:
        raise ValueError("No valid JSON files found in the directory")

    # Convert to DataFrame
    df = pd.DataFrame(all_data)

    # Reorder columns for better readability
    column_order = ['source_file', 'epoch', 'balanced_accuracy', 'val_loss',
                    'val_f1', 'val_AUPRC', 'val_MCC']
    df = df[column_order]

    print(
        f"Processed {len(set(df['source_file']))} files with {len(df)} total epochs")
    return df

In [4]:

def parse_filename(filename):
    """
    Parse a complex filename and extract different components into a dictionary.

    Args:
    filename (str): The input filename to parse

    Returns:
    dict: A dictionary containing parsed components
    """
    # Remove .json extension if present
    filename = filename.replace('.json', '')

    # Split the filename by underscores
    parts = filename.split('_')

    # Initialize a dictionary to store parsed values
    parsed_dict = {}

    # Extract dataset name (first part)
    parsed_dict['dataset_name'] = parts[0]

    # Create a dictionary to store metric values
    metric_values = {}

    if 'no-polyak' in filename:
        parsed_dict['polyak'] = False
    elif 'polyak' in filename:
        parsed_dict['polyak'] = True
    parsed_dict['is_baseline'] = False
    for model in ["logistic_regression", "random_forest", "gradient_boosting", "knn"]:
        if model in filename:
            parsed_dict['is_baseline'] = True

    # Process remaining parts
    for part in parts[1:]:
        # Split each part into metric and value
        metric_match = re.match(r'([a-z]+)(?:-(\d+(?:over\d+)?))?', part)

        if metric_match:
            metric = metric_match.group(1)
            value = metric_match.group(2) if metric_match.group(2) else '0'

            # Convert fractional values like '1over2' to float
            if 'over' in str(value):
                num, denom = value.split('over')
                value = f'{num} / {denom}'
            else:
                value = str(value)

            metric_values[metric] = value

    # Add specific metrics to the dictionary
    metrics_to_extract = ['euclidean', 'chebyshev', 'wasserstein', 'cosine']
    for metric in metrics_to_extract:
        parsed_dict[metric] = metric_values.get(metric, 0)

    return parsed_dict


def parse_filename_column(df, filename_column):
    """
    Apply filename parsing to an entire DataFrame column.

    Args:
    df (pd.DataFrame): Input DataFrame
    filename_column (str): Name of the column containing filenames

    Returns:
    pd.DataFrame: DataFrame with parsed columns added
    """
    # Apply parsing to each filename
    parsed_data = df[filename_column].apply(parse_filename)

    # Convert parsed data to DataFrame
    parsed_df = pd.DataFrame(parsed_data.tolist())

    # Combine original DataFrame with parsed columns
    return pd.concat([df, parsed_df], axis=1)

In [5]:
dfs = process_metrics_jsons(dir_path)

Processed 58 files with 2320 total epochs


In [6]:
def compute_mean_ci(group):
    result = {}
    for col in group.columns:
        if group[col].dtype in ['float64', 'float32']:  # Numeric columns
            mean = group[col].mean()
            sem = group[col].sem()  # Standard error of the mean
            n = len(group[col])
            confidence = 0.95
            t_value = stats.t.ppf((1 + confidence) / 2, n - 1)  # T critical value
            ci = t_value * sem  # Confidence interval
            result[f"{col}_mean"] = mean
            result[f"{col}_ci"] = ci
    return pd.Series(result)

In [7]:
summary = dfs.groupby("source_file").apply(compute_mean_ci).reset_index()

In [8]:
summary

,source_file,balanced_accuracy_mean,balanced_accuracy_ci,val_loss_mean,val_loss_ci,val_f1_mean,val_f1_ci,val_AUPRC_mean,val_AUPRC_ci,val_MCC_mean,val_MCC_ci
0,CICEVSE_Network2024_gradient_boosting.json,0.497249,0.003263,0.000000,0.000000,0.052343,0.004965,0.104050,0.004904,-0.052245,0.002562
1,CICEVSE_Network2024_knn.json,0.724911,0.005494,0.000000,0.000000,0.425803,0.011271,0.375872,0.007839,0.871447,0.003617
2,CICEVSE_Network2024_logistic_regression.json,0.481233,0.004083,0.000000,0.000000,0.017051,0.004648,0.071098,0.002000,-0.067375,0.005924
3,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,0.796456,0.007545,0.684253,0.005999,0.782712,0.039153,0.640831,0.043184,0.745302,0.036754
4,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,0.789633,0.005294,0.675779,0.001656,0.806688,0.023615,0.668950,0.033384,0.769522,0.025860
5,CICEVSE_Network2024_no-polyak_euclidean-1_cheb...,0.788393,0.006838,0.692869,0.003870,0.765556,0.027809,0.608102,0.037699,0.721932,0.030202
6,CICEVSE_Network2024_no-polyak_euclidean-1over3...,0.819206,0.002597,0.674375,0.001285,0.850751,0.004628,0.733309,0.007504,0.818949,0.005329
7,CICEVSE_Network2024_no-polyak_euclidean-1over4...,0.807802,0.003839,0.685952,0.001615,0.809966,0.017938,0.670989,0.026413,0.771867,0.020439
8,CICEVSE_Network2024_polyak_euclidean-0_chebysh...,0.772876,0.008281,0.726851,0.000789,0.742132,0.031123,0.577184,0.042709,0.692637,0.035622
9,CICEVSE_Network2024_polyak_euclidean-0_chebysh...,0.801906,0.003752,0.711774,0.001227,0.805844,0.018607,0.664820,0.026976,0.766504,0.021419


In [9]:
parsed_df = parse_filename_column(summary, 'source_file')

In [10]:
parsed_df['dl_baseline'] = (parsed_df["euclidean"] == "1") & (parsed_df["chebyshev"] == "0") & (parsed_df["wasserstein"] == "0") & (parsed_df["cosine"] == "0")

In [11]:
parsed_df.head(10)

,source_file,balanced_accuracy_mean,balanced_accuracy_ci,val_loss_mean,val_loss_ci,val_f1_mean,val_f1_ci,val_AUPRC_mean,val_AUPRC_ci,val_MCC_mean,val_MCC_ci,dataset_name,is_baseline,euclidean,chebyshev,wasserstein,cosine,polyak,dl_baseline
0,CICEVSE_Network2024_gradient_boosting.json,0.497249,0.003263,0.000000,0.000000,0.052343,0.004965,0.104050,0.004904,-0.052245,0.002562,CICEVSE,True,0,0,0,0,NaN,False
1,CICEVSE_Network2024_knn.json,0.724911,0.005494,0.000000,0.000000,0.425803,0.011271,0.375872,0.007839,0.871447,0.003617,CICEVSE,True,0,0,0,0,NaN,False
2,CICEVSE_Network2024_logistic_regression.json,0.481233,0.004083,0.000000,0.000000,0.017051,0.004648,0.071098,0.002000,-0.067375,0.005924,CICEVSE,True,0,0,0,0,NaN,False
3,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,0.796456,0.007545,0.684253,0.005999,0.782712,0.039153,0.640831,0.043184,0.745302,0.036754,CICEVSE,False,0,1 / 2,1 / 2,0,False,False
4,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,0.789633,0.005294,0.675779,0.001656,0.806688,0.023615,0.668950,0.033384,0.769522,0.025860,CICEVSE,False,0,1 / 3,1 / 3,1 / 3,False,False
5,CICEVSE_Network2024_no-polyak_euclidean-1_cheb...,0.788393,0.006838,0.692869,0.003870,0.765556,0.027809,0.608102,0.037699,0.721932,0.030202,CICEVSE,False,1,0,0,0,False,True
6,CICEVSE_Network2024_no-polyak_euclidean-1over3...,0.819206,0.002597,0.674375,0.001285,0.850751,0.004628,0.733309,0.007504,0.818949,0.005329,CICEVSE,False,1 / 3,1 / 3,0,1 / 3,False,False
7,CICEVSE_Network2024_no-polyak_euclidean-1over4...,0.807802,0.003839,0.685952,0.001615,0.809966,0.017938,0.670989,0.026413,0.771867,0.020439,CICEVSE,False,1 / 4,1 / 4,1 / 4,1 / 4,False,False
8,CICEVSE_Network2024_polyak_euclidean-0_chebysh...,0.772876,0.008281,0.726851,0.000789,0.742132,0.031123,0.577184,0.042709,0.692637,0.035622,CICEVSE,False,0,1 / 2,1 / 2,0,True,False
9,CICEVSE_Network2024_polyak_euclidean-0_chebysh...,0.801906,0.003752,0.711774,0.001227,0.805844,0.018607,0.664820,0.026976,0.766504,0.021419,CICEVSE,False,0,1 / 3,1 / 3,1 / 3,True,False


### Baseline Results

In [12]:
baseline_df = parsed_df[parsed_df['is_baseline'] == True]

In [13]:
baseline_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 16 entries, 0 to 57
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   source_file             16 non-null     object 
 1   balanced_accuracy_mean  16 non-null     float64
 2   balanced_accuracy_ci    16 non-null     float64
 3   val_loss_mean           16 non-null     float64
 4   val_loss_ci             16 non-null     float64
 5   val_f1_mean             16 non-null     float64
 6   val_f1_ci               16 non-null     float64
 7   val_AUPRC_mean          16 non-null     float64
 8   val_AUPRC_ci            16 non-null     float64
 9   val_MCC_mean            16 non-null     float64
 10  val_MCC_ci              16 non-null     float64
 11  dataset_name            16 non-null     object 
 12  is_baseline             16 non-null     bool   
 13  euclidean               16 non-null     object 
 14  chebyshev               16 non-null     object 
 

In [14]:
baseline_table_df = baseline_df.sort_values('balanced_accuracy_mean', ascending=False)\
    .drop(columns=['is_baseline', 'polyak', 'val_loss_mean', 'val_loss_ci', 'euclidean', 'chebyshev', 'wasserstein', 'cosine','val_MCC_mean', 'val_MCC_ci'])\
    .sort_values(['dataset_name', 'balanced_accuracy_mean'], ascending=[True, False]) \
    .groupby('dataset_name') \
    .head(1)

In [15]:
baseline_table_df

,source_file,balanced_accuracy_mean,balanced_accuracy_ci,val_f1_mean,val_f1_ci,val_AUPRC_mean,val_AUPRC_ci,dataset_name,dl_baseline
13,CICEVSE_Network2024_random_forest.json,0.821173,0.007938,0.640275,0.018729,0.567085,0.018670,CICEVSE,False
26,CICIDS2017_gradient_boosting.json,0.721114,0.011796,0.407909,0.020490,0.364668,0.016953,CICIDS2017,False
44,CICIoV2024_logistic_regression.json,0.979838,0.013769,0.962288,0.025717,0.962450,0.025604,CICIoV2024,False


In [16]:
ordered_baseline = baseline_table_df[['dataset_name', 'source_file', 'balanced_accuracy_mean', 'balanced_accuracy_ci', 'val_f1_mean','val_f1_ci','val_AUPRC_mean','val_AUPRC_ci']].copy()

In [17]:
ordered_baseline

,dataset_name,source_file,balanced_accuracy_mean,balanced_accuracy_ci,val_f1_mean,val_f1_ci,val_AUPRC_mean,val_AUPRC_ci
13,CICEVSE,CICEVSE_Network2024_random_forest.json,0.821173,0.007938,0.640275,0.018729,0.567085,0.018670
26,CICIDS2017,CICIDS2017_gradient_boosting.json,0.721114,0.011796,0.407909,0.020490,0.364668,0.016953
44,CICIoV2024,CICIoV2024_logistic_regression.json,0.979838,0.013769,0.962288,0.025717,0.962450,0.025604


### Polyak stuff

In [18]:
non_baseline_results = parsed_df[~parsed_df['is_baseline'] & ~parsed_df['dl_baseline']].copy()

In [19]:
non_baseline_results.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 3 to 56
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   source_file             36 non-null     object 
 1   balanced_accuracy_mean  36 non-null     float64
 2   balanced_accuracy_ci    36 non-null     float64
 3   val_loss_mean           36 non-null     float64
 4   val_loss_ci             36 non-null     float64
 5   val_f1_mean             36 non-null     float64
 6   val_f1_ci               36 non-null     float64
 7   val_AUPRC_mean          36 non-null     float64
 8   val_AUPRC_ci            36 non-null     float64
 9   val_MCC_mean            36 non-null     float64
 10  val_MCC_ci              36 non-null     float64
 11  dataset_name            36 non-null     object 
 12  is_baseline             36 non-null     bool   
 13  euclidean               36 non-null     object 
 14  chebyshev               36 non-null     object 
 

In [20]:
non_baseline_results.head(10)

,source_file,balanced_accuracy_mean,balanced_accuracy_ci,val_loss_mean,val_loss_ci,val_f1_mean,val_f1_ci,val_AUPRC_mean,val_AUPRC_ci,val_MCC_mean,val_MCC_ci,dataset_name,is_baseline,euclidean,chebyshev,wasserstein,cosine,polyak,dl_baseline
3,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,0.796456,0.007545,0.684253,0.005999,0.782712,0.039153,0.640831,0.043184,0.745302,0.036754,CICEVSE,False,0,1 / 2,1 / 2,0,False,False
4,CICEVSE_Network2024_no-polyak_euclidean-0_cheb...,0.789633,0.005294,0.675779,0.001656,0.806688,0.023615,0.668950,0.033384,0.769522,0.025860,CICEVSE,False,0,1 / 3,1 / 3,1 / 3,False,False
6,CICEVSE_Network2024_no-polyak_euclidean-1over3...,0.819206,0.002597,0.674375,0.001285,0.850751,0.004628,0.733309,0.007504,0.818949,0.005329,CICEVSE,False,1 / 3,1 / 3,0,1 / 3,False,False
7,CICEVSE_Network2024_no-polyak_euclidean-1over4...,0.807802,0.003839,0.685952,0.001615,0.809966,0.017938,0.670989,0.026413,0.771867,0.020439,CICEVSE,False,1 / 4,1 / 4,1 / 4,1 / 4,False,False
8,CICEVSE_Network2024_polyak_euclidean-0_chebysh...,0.772876,0.008281,0.726851,0.000789,0.742132,0.031123,0.577184,0.042709,0.692637,0.035622,CICEVSE,False,0,1 / 2,1 / 2,0,True,False
9,CICEVSE_Network2024_polyak_euclidean-0_chebysh...,0.801906,0.003752,0.711774,0.001227,0.805844,0.018607,0.664820,0.026976,0.766504,0.021419,CICEVSE,False,0,1 / 3,1 / 3,1 / 3,True,False
11,CICEVSE_Network2024_polyak_euclidean-1over3_ch...,0.814350,0.002857,0.712984,0.001063,0.835176,0.010414,0.708854,0.015927,0.801173,0.011688,CICEVSE,False,1 / 3,1 / 3,0,1 / 3,True,False
12,CICEVSE_Network2024_polyak_euclidean-1over4_ch...,0.793312,0.005799,0.712891,0.000812,0.809750,0.025759,0.674704,0.034110,0.771098,0.029929,CICEVSE,False,1 / 4,1 / 4,1 / 4,1 / 4,True,False
17,CICEVSE_PowerB2024_no-polyak_euclidean-0_cheby...,0.689812,0.002885,0.665278,0.002582,0.451526,0.004940,0.282462,0.003742,0.361903,0.005028,CICEVSE,False,0,1 / 2,1 / 2,0,False,False
18,CICEVSE_PowerB2024_no-polyak_euclidean-0_cheby...,0.708060,0.003411,0.645514,0.001147,0.475553,0.004753,0.301287,0.003858,0.391466,0.005450,CICEVSE,False,0,1 / 3,1 / 3,1 / 3,False,False


In [21]:
non_baseline_results_table = non_baseline_results.sort_values(['dataset_name','balanced_accuracy_mean'], ascending=[True, False]) \
    .groupby(['dataset_name', 'polyak']) \
    .head(1) \
    .drop(columns=['is_baseline'])

In [22]:
column_order =[
 'dataset_name',
 'polyak',
 'balanced_accuracy_mean',
 'balanced_accuracy_ci',
 'val_f1_mean',
 'val_f1_ci',
 'val_AUPRC_mean',
 'val_AUPRC_ci',
 'euclidean',
 'chebyshev',
 'wasserstein',
 'cosine'
]

In [23]:
list(non_baseline_results_table.columns)

['source_file',
 'balanced_accuracy_mean',
 'balanced_accuracy_ci',
 'val_loss_mean',
 'val_loss_ci',
 'val_f1_mean',
 'val_f1_ci',
 'val_AUPRC_mean',
 'val_AUPRC_ci',
 'val_MCC_mean',
 'val_MCC_ci',
 'dataset_name',
 'euclidean',
 'chebyshev',
 'wasserstein',
 'cosine',
 'polyak',
 'dl_baseline']

In [24]:
ordered_results = non_baseline_results_table[column_order].copy()

## LATEX Stuff


Utils for updating the LaTex tables dynamically


In [25]:
def dataframe_to_latex_escape(df, caption="", label="", column_format=None, fit_single_column=False):
    """
    Converts a Pandas DataFrame to a LaTeX table, escaping special characters and ignoring the index.

    Parameters:
        df (pd.DataFrame): The DataFrame to convert.
        caption (str): The caption for the table.
        label (str): The label for the table (used for referencing in LaTeX).
        column_format (str): Optional LaTeX column format string (e.g., "lccccccc").
        fit_single_column (bool): If True, resize the table to fit a single column.

    Returns:
        str: The LaTeX string for the table.
    """
    # Escape underscores and other LaTeX special characters in column names
    df.columns = [col.replace('_', r'\_') for col in df.columns]

    # Escape underscores and other special characters in data
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].apply(lambda x: str(x).replace('_', r'\_'))

    # Define default column alignment
    if column_format is None:
        column_format = "l" + "c" * (df.shape[1] - 1)

    # Convert the DataFrame to a LaTeX table without the index
    latex_table = df.to_latex(
        index=False,  # Ignore the index
        header=True,
        column_format=column_format,
        float_format="{:.4f}".format,
        escape=False  # Disable escape as we are escaping manually
    )

    # Wrap table with resizing if required
    if fit_single_column:
        latex_table = (
            "\\begin{table}[h]\n"
            "\\centering\n"
            "\\resizebox{\\columnwidth}{!}{%\n"
            + latex_table +
            "}\n"
            f"\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n"
            "\\end{table}"
        )
    else:
        latex_table = (
            "\\begin{table*}[h]\n"
            "\\centering\n"
            + latex_table +
            f"\\caption{{{caption}}}\n"
            f"\\label{{{label}}}\n"
            "\\end{table*}"
        )

    return latex_table

### Latex for resuts

In [26]:
# Or to print directly
latex_output = dataframe_to_latex_escape(ordered_results)
print(latex_output)

\begin{table*}[h]
\centering
\begin{tabular}{lccccccccccc}
\toprule
dataset\_name & polyak & balanced\_accuracy\_mean & balanced\_accuracy\_ci & val\_f1\_mean & val\_f1\_ci & val\_AUPRC\_mean & val\_AUPRC\_ci & euclidean & chebyshev & wasserstein & cosine \\
\midrule
CICEVSE & False & 0.8192 & 0.0026 & 0.8508 & 0.0046 & 0.7333 & 0.0075 & 1 / 3 & 1 / 3 & 0 & 1 / 3 \\
CICEVSE & True & 0.8143 & 0.0029 & 0.8352 & 0.0104 & 0.7089 & 0.0159 & 1 / 3 & 1 / 3 & 0 & 1 / 3 \\
CICIDS2017 & True & 0.8768 & 0.0054 & 0.6626 & 0.0169 & 0.4658 & 0.0208 & 1 / 2 & 0 & 0 & 1 / 2 \\
CICIDS2017 & False & 0.8740 & 0.0074 & 0.6727 & 0.0201 & 0.4798 & 0.0254 & 1 / 3 & 1 / 3 & 0 & 1 / 3 \\
CICIoV2024 & False & 0.9813 & 0.0074 & 0.9718 & 0.0124 & 0.9506 & 0.0210 & 1 / 4 & 1 / 4 & 1 / 4 & 1 / 4 \\
CICIoV2024 & True & 0.9718 & 0.0093 & 0.9404 & 0.0228 & 0.8992 & 0.0370 & 1 / 4 & 1 / 4 & 1 / 4 & 1 / 4 \\
\bottomrule
\end{tabular}
\caption{}
\label{}
\end{table*}


### Baseline

### DL

In [27]:
dl_baseline_results = parsed_df[~parsed_df['is_baseline'] & parsed_df['dl_baseline']].copy()

In [28]:
dl_baseline_results[column_order]

,dataset_name,polyak,balanced_accuracy_mean,balanced_accuracy_ci,val_f1_mean,val_f1_ci,val_AUPRC_mean,val_AUPRC_ci,euclidean,chebyshev,wasserstein,cosine
5,CICEVSE,False,0.788393,0.006838,0.765556,0.027809,0.608102,0.037699,1,0,0,0
10,CICEVSE,True,0.788833,0.006264,0.756786,0.028485,0.595661,0.037432,1,0,0,0
31,CICIDS2017,False,0.846050,0.010092,0.598169,0.022519,0.391342,0.025282,1,0,0,0
37,CICIDS2017,True,0.857176,0.006581,0.605207,0.026173,0.401005,0.029107,1,0,0,0
47,CICIoV2024,False,0.966923,0.009572,0.959054,0.010560,0.927672,0.017949,1,0,0,0
53,CICIoV2024,True,0.962140,0.010742,0.899853,0.033783,0.837306,0.048910,1,0,0,0


In [29]:
print(dataframe_to_latex_escape(dl_baseline_results[column_order].copy()))

\begin{table*}[h]
\centering
\begin{tabular}{lccccccccccc}
\toprule
dataset\_name & polyak & balanced\_accuracy\_mean & balanced\_accuracy\_ci & val\_f1\_mean & val\_f1\_ci & val\_AUPRC\_mean & val\_AUPRC\_ci & euclidean & chebyshev & wasserstein & cosine \\
\midrule
CICEVSE & False & 0.7884 & 0.0068 & 0.7656 & 0.0278 & 0.6081 & 0.0377 & 1 & 0 & 0 & 0 \\
CICEVSE & True & 0.7888 & 0.0063 & 0.7568 & 0.0285 & 0.5957 & 0.0374 & 1 & 0 & 0 & 0 \\
CICIDS2017 & False & 0.8460 & 0.0101 & 0.5982 & 0.0225 & 0.3913 & 0.0253 & 1 & 0 & 0 & 0 \\
CICIDS2017 & True & 0.8572 & 0.0066 & 0.6052 & 0.0262 & 0.4010 & 0.0291 & 1 & 0 & 0 & 0 \\
CICIoV2024 & False & 0.9669 & 0.0096 & 0.9591 & 0.0106 & 0.9277 & 0.0179 & 1 & 0 & 0 & 0 \\
CICIoV2024 & True & 0.9621 & 0.0107 & 0.8999 & 0.0338 & 0.8373 & 0.0489 & 1 & 0 & 0 & 0 \\
\bottomrule
\end{tabular}
\caption{}
\label{}
\end{table*}


### Non DL

In [31]:
ordered_baseline

,dataset\_name,source\_file,balanced\_accuracy\_mean,balanced\_accuracy\_ci,val\_f1\_mean,val\_f1\_ci,val\_AUPRC\_mean,val\_AUPRC\_ci
13,CICEVSE,CICEVSE\_Network2024\_random\_forest.json,0.821173,0.007938,0.640275,0.018729,0.567085,0.018670
26,CICIDS2017,CICIDS2017\_gradient\_boosting.json,0.721114,0.011796,0.407909,0.020490,0.364668,0.016953
44,CICIoV2024,CICIoV2024\_logistic\_regression.json,0.979838,0.013769,0.962288,0.025717,0.962450,0.025604


In [30]:
latex_output = dataframe_to_latex_escape(ordered_baseline)
print(latex_output)

\begin{table*}[h]
\centering
\begin{tabular}{lccccccc}
\toprule
dataset\_name & source\_file & balanced\_accuracy\_mean & balanced\_accuracy\_ci & val\_f1\_mean & val\_f1\_ci & val\_AUPRC\_mean & val\_AUPRC\_ci \\
\midrule
CICEVSE & CICEVSE\_Network2024\_random\_forest.json & 0.8212 & 0.0079 & 0.6403 & 0.0187 & 0.5671 & 0.0187 \\
CICIDS2017 & CICIDS2017\_gradient\_boosting.json & 0.7211 & 0.0118 & 0.4079 & 0.0205 & 0.3647 & 0.0170 \\
CICIoV2024 & CICIoV2024\_logistic\_regression.json & 0.9798 & 0.0138 & 0.9623 & 0.0257 & 0.9625 & 0.0256 \\
\bottomrule
\end{tabular}
\caption{}
\label{}
\end{table*}


## CICIoV


In [45]:
def read_csv_files_in_folder(folder_path):
    """
    Reads CSV files from a folder and concatenates them into a single pandas DataFrame.

    Parameters:
    folder_path (str): Path to the folder containing the CSV files

    Returns:
    pandas.DataFrame: Concatenated DataFrame from all CSV files in the folder
    """
    # Get a list of all CSV files in the folder
    csv_files = [os.path.join(folder_path, f)
                 for f in os.listdir(folder_path) if f.endswith('.csv')]

    # Read each CSV file and append to a list of DataFrames
    dfs = []
    for csv_file in csv_files:
        df = pd.read_csv(csv_file, low_memory=False)
        df['target'] = df['specific_class'].str.lower()
        dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    return combined_df

In [34]:
data = pd.read_parquet('./data/parquets/cicevse_network.parquet')

In [35]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482309 entries, 0 to 482308
Data columns (total 88 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            482309 non-null  int64  
 1   expiration_id                 482309 non-null  int64  
 2   src_ip                        482309 non-null  object 
 3   src_mac                       482309 non-null  object 
 4   src_oui                       482309 non-null  object 
 5   src_port                      482309 non-null  int64  
 6   dst_ip                        482309 non-null  object 
 7   dst_mac                       482309 non-null  object 
 8   dst_oui                       482309 non-null  object 
 9   dst_port                      482309 non-null  int64  
 10  protocol                      482309 non-null  int64  
 11  ip_version                    482309 non-null  int64  
 12  vlan_id                       482309 non-nul

In [ ]:
data.head().iloc[:, 10:]

,protocol,ip_version,vlan_id,tunnel_id,bidirectional_first_seen_ms,bidirectional_last_seen_ms,bidirectional_duration_ms,bidirectional_packets,bidirectional_bytes,src2dst_first_seen_ms,...,application_category_name,application_is_guessed,application_confidence,requested_server_name,client_fingerprint,server_fingerprint,user_agent,content_type,target,state
0,6,4,0,0,1703261991666,1703261991779,113,2,120,1703261991666,...,Unspecified,0,0,None,None,None,None,None,aggressive-scan,charging
1,6,4,0,0,1703261991667,1703261991784,117,2,120,1703261991667,...,Email,1,1,None,None,None,None,None,aggressive-scan,charging
2,6,4,0,0,1703261991667,1703261991784,117,2,120,1703261991667,...,System,1,1,None,None,None,None,None,aggressive-scan,charging
3,6,4,0,0,1703261991667,1703261991785,118,2,120,1703261991667,...,Email,1,1,None,None,None,None,None,aggressive-scan,charging
4,6,4,0,0,1703261991667,1703261991785,118,2,120,1703261991667,...,RPC,1,1,None,None,None,None,None,aggressive-scan,charging


In [ ]:
def read_ciciov(data_path, target):
    '''
    Read parquet file and return the dataset features and target separated.
    '''
    df = pd.read_parquet(data_path)
    x = df.drop(target, axis=1).iloc[:, 1:-3]
    y = df[target]
    return x, y

In [ ]:
def read_cicevse(data_path, target):
    '''
    Read parquet file and return the dataset features and target separated.
    '''
    df = pd.read_parquet(data_path)
    x = df.drop(target, axis=1).iloc[:, 14:-6]
    y = df[target]
    return x, y

In [46]:
def read_data(data_path, target):
    '''
    Read parquet file and return the dataset features and target separated.
    '''
    df = pd.read_parquet(data_path)
    x = df.drop(target, axis=1)
    y = df[target]
    return x, y

In [ ]:
def preprocess_data(X, y, features, batch_size):
    scaler = sklearn.preprocessing.RobustScaler()

    X_scaled = scaler.fit_transform(X[features].values)
    X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
    y_tensor = torch.tensor(y.values, dtype=torch.float32)
    n_features = X_scaled.shape[1]

    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=train_batch_size, shuffle=True)

    return X_tensor, y_tensor, dataloader

In [3]:
X, Y = read_ciciov('./data/parquets/ciciov.parquet', 'target')

In [4]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1408219 entries, 0 to 1408218
Data columns (total 8 columns):
 #   Column  Non-Null Count    Dtype
---  ------  --------------    -----
 0   DATA_0  1408219 non-null  int64
 1   DATA_1  1408219 non-null  int64
 2   DATA_2  1408219 non-null  int64
 3   DATA_3  1408219 non-null  int64
 4   DATA_4  1408219 non-null  int64
 5   DATA_5  1408219 non-null  int64
 6   DATA_6  1408219 non-null  int64
 7   DATA_7  1408219 non-null  int64
dtypes: int64(8)
memory usage: 86.0 MB


In [58]:
Y

0                  benign
1                  benign
2                  benign
3                  benign
4                  benign
                ...      
1408214    steering_wheel
1408215    steering_wheel
1408216    steering_wheel
1408217    steering_wheel
1408218    steering_wheel
Name: target, Length: 1408219, dtype: object

In [47]:
X, Y = read_cicevse('./data/parquets/cicevse_network.parquet', 'target')

In [48]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 482309 entries, 0 to 482308
Data columns (total 67 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   bidirectional_first_seen_ms   482309 non-null  int64  
 1   bidirectional_last_seen_ms    482309 non-null  int64  
 2   bidirectional_duration_ms     482309 non-null  int64  
 3   bidirectional_packets         482309 non-null  int64  
 4   bidirectional_bytes           482309 non-null  int64  
 5   src2dst_first_seen_ms         482309 non-null  int64  
 6   src2dst_last_seen_ms          482309 non-null  int64  
 7   src2dst_duration_ms           482309 non-null  int64  
 8   src2dst_packets               482309 non-null  int64  
 9   src2dst_bytes                 482309 non-null  int64  
 10  dst2src_first_seen_ms         482309 non-null  int64  
 11  dst2src_last_seen_ms          482309 non-null  int64  
 12  dst2src_duration_ms           482309 non-nul

In [3]:
results = []
raw_results = {}
for file in os.listdir('results'):
    if file.endswith('.json'):
        file_path = os.path.join('results', file)
        with open(file_path, 'r') as f:
            data = json.load(f)
            results_arrays = {}
            results_arrays['Model'] = file
            results_arrays['Accuracy'] = np.mean(data['balanced_accuracy'])
            results_arrays['AUPRC'] = np.mean(data['val_AUPRC'])
            results_arrays['MCC'] = np.mean(data['val_MCC'])
            results_arrays['F1'] = np.mean(data['val_f1'])
            raw_results[file] = data
            results.append(results_arrays)
            # results_arrays['Loss'] = np.mean(data['val_loss'])

# # 3. Calculate mean of balanced_accuracy
# balanced_accuracies = [result['balanced_accuracy'] for result in results_arrays]
# mean_accuracy = np.mean(balanced_accuracies)

# print(f"Mean balanced accuracy: {mean_accuracy:.4f}")

In [5]:
baseline = 'no-polyak_euclidean-1_chebyshev-0_cosine-0_wasserstein-0.json'

In [ ]:

import numpy as np


def perform_ttest(group_a, group_b, alpha=0.05):
    mean_a = np.mean(group_a)
    mean_b = np.mean(group_b)
    std_a = np.std(group_a, ddof=1)  # ddof=1 for sample standard deviation
    std_b = np.std(group_b, ddof=1)

    t_stat, p_value = stats.ttest_ind(group_a, group_b)

    print(f"Group A - Mean: {mean_a:.4f}, Std: {std_a:.4f}")
    print(f"Group B - Mean: {mean_b:.4f}, Std: {std_b:.4f}")
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    if p_value < alpha:
        print("The difference is statistically significant (p < 0.05)")
    else:
        print("The difference is not statistically significant (p >= 0.05)")


perform_ttest(raw_results[baseline]['balanced_accuracy'],
              raw_results['polyak_euclidean-1over4_chebyshev-1over4_cosine-1over4_wasserstein-1over4.json']['balanced_accuracy'])

Group A - Mean: 0.8460, Std: 0.0316
Group B - Mean: 0.8700, Std: 0.0274
t-statistic: -3.6303
p-value: 0.0005
The difference is statistically significant (p < 0.05)


In [ ]:
pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)  # 200, att

,Model,Accuracy,AUPRC,MCC,F1
15,polyak_euclidean-1over2_chebyshev-0_cosine-1ov...,0.876787,0.465844,0.634678,0.662575
1,no-polyak_euclidean-1over3_chebyshev-1over3_co...,0.873976,0.479766,0.641218,0.672710
9,polyak_euclidean-0_chebyshev-1over3_cosine-1ov...,0.872501,0.451537,0.613864,0.649868
10,polyak_euclidean-1over3_chebyshev-1over3_cosin...,0.871558,0.464902,0.629037,0.659325
11,polyak_euclidean-1over4_chebyshev-1over4_cosin...,0.870034,0.415222,0.582885,0.617276
7,no-polyak_euclidean-1over2_chebyshev-0_cosine-...,0.870002,0.500339,0.656027,0.688587
0,no-polyak_euclidean-0_chebyshev-1over3_cosine-...,0.869473,0.459097,0.624010,0.656282
3,no-polyak_euclidean-1over4_chebyshev-1over4_co...,0.866381,0.448133,0.610248,0.646599
13,polyak_euclidean-1_chebyshev-0_cosine-0_wasser...,0.857176,0.401005,0.579437,0.605207
8,no-polyak_euclidean-1_chebyshev-0_cosine-0_was...,0.846050,0.391342,0.576802,0.598169


In [ ]:
200